# 01 — Data Collection

This notebook documents Phase 1 of the ChatTLA research project: building a verified TLA+ training corpus.

## Overview

We build the corpus in six stages:
1. **Seed ingestion** — 205 MIT-licensed specs from FormaLLM submodule
2. **GitHub scraping** — TLA+ code search via GitHub API (Tier-2 queries)
3. **TLC/SANY validation** — Every spec is checked; assigned gold/silver/bronze tier
4. **Deduplication** — MinHash LSH at Jaccard ≥ 0.8 removes near-duplicate specs
5. **Annotation** — Local Ollama gpt-oss:20b generates NL descriptions (zero cost)
6. **Combine** — Gold + silver specs merged into `data/validated/combined.jsonl`

**Primary data quality metric**: fraction of gold-tier records (TLC-verified, no violations).
**Target**: ≥10,000 gold, ≥50,000 silver total.

In [3]:
import sys
from pathlib import Path

# Add repo root to path so we can import src module
REPO_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)
print('Python path set for imports')

Repo root: /home/espencer2/ChatTLA
Python path set for imports


## Step 1: Ingest FormaLLM Seed Corpus

FormaLLM provides 205 expert-curated TLA+ specs with pre-existing natural-language annotations.
These are the highest-quality seed examples and are never deduplicated against — they are the anchor.

**Format**: each spec is a `DatasetRecord` with `.tla_content`, `.cfg_content`, and `.annotation.natural_language_description` pre-populated.

In [4]:
from src.scraper.ingest_formalllm import run as ingest_formalllm
n = ingest_formalllm(overwrite=True)
print(f'Ingested {n} FormaLLM records → data/raw/formalllm.jsonl')

[ingest_formalllm] WARNING: missing /home/espencer2/ChatTLA/data/FormaLLM/data/aba_asyn_byz/tla/aba_asyn_byz.tla, skipping
[ingest_formalllm] WARNING: missing /home/espencer2/ChatTLA/data/FormaLLM/data/ACP_NB/tla/ACP_NB.tla, skipping
[ingest_formalllm] WARNING: missing /home/espencer2/ChatTLA/data/FormaLLM/data/ACP_NB_TLC/tla/ACP_NB_TLC.tla, skipping
[ingest_formalllm] WARNING: missing /home/espencer2/ChatTLA/data/FormaLLM/data/ACP_NB_WRONG_TLC/tla/ACP_NB_WRONG_TLC.tla, skipping
[ingest_formalllm] WARNING: missing /home/espencer2/ChatTLA/data/FormaLLM/data/ACP_SB/tla/ACP_SB.tla, skipping
[ingest_formalllm] WARNING: missing /home/espencer2/ChatTLA/data/FormaLLM/data/ACP_SB_TLC/tla/ACP_SB_TLC.tla, skipping
[ingest_formalllm] WARNING: missing /home/espencer2/ChatTLA/data/FormaLLM/data/AllocatorImplementation/tla/AllocatorImplementation.tla, skipping
[ingest_formalllm] WARNING: missing /home/espencer2/ChatTLA/data/FormaLLM/data/AllocatorRefinement/tla/AllocatorRefinement.tla, skipping
[ing

In [5]:
# Inspect seed corpus
seed_records = []
formalllm_jsonl = REPO_ROOT / 'data' / 'raw' / 'formalllm.jsonl'
for line in formalllm_jsonl.open():
    seed_records.append(json.loads(line))

seed_df = pd.DataFrame([
    {
        'module': r['metadata']['module_name'],
        'has_cfg': r['cfg_content'] is not None,
        'has_annotation': bool(r.get('annotation', {}) and r['annotation'].get('natural_language_description'))
    }
    for r in seed_records
])

print(f'Total: {len(seed_df)}')
print(f'Has .cfg: {seed_df.has_cfg.sum()}')
print(f'Has annotation: {seed_df.has_annotation.sum()}')
seed_df.head(10)

Total: 35
Has .cfg: 17
Has annotation: 34


,module,has_cfg,has_annotation
0,bcastByz,True,True
1,bcastFolklore,True,True
2,bosco,True,True
3,c1cs,True,True
4,CarTalkPuzzle,False,True
5,cbc_max,False,True
6,Chameneos,True,True
7,CheckpointCoordination,False,True
8,CigaretteSmokers,True,True
9,CoffeeCan,False,True


## Step 2: Run Validation on Seed Corpus

Before running any scraping, let's validate the FormaLLM seed to understand the quality baseline.

**Expected**: most FormaLLM specs are already gold-tier (they come from official TLA+ examples repos).

**Research note**: A SANY parse rate < 90% on FormaLLM would indicate a problem with our validator setup (e.g., wrong Java version, missing tla2tools.jar).

In [6]:
# Run pipeline in dry-run mode (seed + validate only)
from src.scraper.pipeline import stage_validate

formalllm_jsonl = REPO_ROOT / 'data' / 'raw' / 'formalllm.jsonl'
validated_path, rejected_path = stage_validate([formalllm_jsonl])

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
# Analyse validation results
val_records = []
for line in validated_path.open():
    r = json.loads(line)
    tlc = r.get('tlc_result') or {}
    qual = r.get('quality') or {}
    val_records.append({
        'module': r['metadata']['module_name'],
        'tier': tlc.get('tier', 'unknown'),
        'quality_overall': qual.get('overall', 0),
        'line_count': qual.get('line_count', 0),
        'has_invariants': qual.get('has_invariants', False),
    })

val_df = pd.DataFrame(val_records)
tier_counts = val_df['tier'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tier_counts.plot(kind='bar', ax=axes[0], title='Validation Tiers (FormaLLM seed)')
axes[0].set_xlabel('Tier')
axes[0].set_ylabel('Count')

val_df['quality_overall'].value_counts().sort_index().plot(kind='bar', ax=axes[1], title='Quality Score Distribution')
axes[1].set_xlabel('Quality Score (1-5)')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'outputs' / 'seed_validation_stats.png', dpi=150)
plt.show()
print(tier_counts)

## Step 3: Full Pipeline Run

The full pipeline (GitHub scraping + validation + dedup + annotation) is long-running.
Run it via CLI — it checkpoints at each stage:

```bash
# FormaLLM seed only (fast — use for CI/smoke testing):
python -m src.scraper.pipeline --dry-run

# Full scrape (several hours, requires GITHUB_TOKEN_1 in .env):
python -m src.scraper.pipeline

# No annotation (if no Ollama available):
python -m src.scraper.pipeline --no-github --no-annotate
```

After the pipeline completes, reload this cell to analyse the combined corpus.

In [ ]:
combined = REPO_ROOT / 'data' / 'validated' / 'combined.jsonl'
if combined.exists():
    combined_records = [json.loads(l) for l in combined.open() if l.strip()]
    df = pd.DataFrame([
        {
            'source_type': r['source'].split(':')[0],
            'tier': (r.get('tlc_result') or {}).get('tier', 'unknown'),
            'domain': (r.get('annotation') or {}).get('domain', 'unknown'),
            'quality': ((r.get('quality') or {}).get('overall', 0)),
            'difficulty': (r.get('annotation') or {}).get('difficulty', 0),
        }
        for r in combined_records
    ])
    print(f'Total records: {len(df)}')
    print(f"Tier breakdown:\n{df['tier'].value_counts()}")
    print(f"\nDomain breakdown:\n{df['domain'].value_counts()}")
else:
    print('combined.jsonl not yet generated — run the pipeline first.')